# Processing Template v1.0.0
___
Jonathan Zhang

Here, we outline how to use this notebook to process stitched and background-subtracted images from HT-BAM experiments. Before you start, carefully read the guidelines below.

## Processing logic



In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import htbam_analysis.processing as processing
from pathlib import Path
import pandas as pd

import json

from typing import Union, Tuple

In [ ]:
# import gc
# import warnings
# from copy import deepcopy
# from collections import namedtuple
# # from htbam_analysis.processing import experiment

# import numpy as np
# import numpy.ma as ma
# from tqdm import tqdm
# import pandas as pd

# from typing import List
# import multiprocessing

# import cv2
# import skimage
# from skimage import io

# class ChipImage:
#     stampWidth = 100
#     # stampWidth = 60

#     def __init__(
#         self, device, raster, corners, attrs=None
#     ):
#         """
#         Constructor for a ChipImage object. A ChipImage represents a single rastered chip image
#         and all associated information.

#         Arguments:
#             (experiment.Device) device: device object
#             (str | pathlib.Path) raster: path of the rasted image file
#             (dict) ids: dict of ChipImage identifiers and values (i.e., time, concentration, etc.)
#             (tuple | namedtuple) corners: cornerpositions of the form
#                 ((ULx, ULy),(URx, URy),(LLx, LLy),(LRx, LRy))
#             (pd.DataFrame) pinlist:
#             (str) channel: imaging channel
#             (int) exposure: exposure time (ms)
#             (dict) attrs: arbitrary metadata

#         Returns:
#             None

#         """

#         self.device = device
    
#         self.data_ref = raster  # reference
#         # self.ids = ids
#         self.stampWidth = ChipImage.stampWidth
#         self.pinlist = device.pinlist
#         self.attrs = attrs
#         self.stamps = None
#         self.centers = []

#         if not isinstance(corners, type(namedtuple)):
#             chipCorners = namedtuple("Corners", ["ul", "ur", "bl", "br"])
#             self.corners = chipCorners(*corners)
#         else:
#             self.corners = corners

#         self.__grid()

#     def __grid(self, altCorners=None):
#         """
#         Calculates chamber center positions interpolating the corner positions

#         Arguments:
#             (tuple | namedtuple) corners: cornerpositions of the form
#                 ((ULx, ULy),(URx, URy),(LLx, LLy),(LRx, LRy))

#         Returns:
#             None

#         """

#         if altCorners:
#             if not isinstance(altCorners, namedtuple):
#                 chipCorners = namedtuple("Corners", ["ul", "ur", "bl", "br"])
#                 corners = chipCorners(*altCorners)
#             else:
#                 corners = altCorners
#             self.centers = self.quadrilateralInterp(altCorners, self.device.dims)
#         self.centers = self.quadrilateralInterp(self.corners, self.device.dims)

#     def stamp(self):
#         """
#         Wrapper class for chamber stamping. Stamps ChipImage using calculated center positions.

#         Arguments:
#             None

#         Returns:
#             None

#         """
#         self.stamps = self._stamp()

#     def _stamp(self):
#         """
#         Stamps the chipImage using calculated center positions.

#         Arguments:
#             None

#         Returns:
#             (np.ndarray) an array of Stamp objects

#         """

#         pinlistFeatureTitle = "MutantID"
#         img = skimage.io.imread(self.data_ref)

#         def makeSlice(center, width):
#             s = (
#                 slice(center[1] - (width // 2), center[1] + (width // 2)),
#                 slice(center[0] - (width // 2), center[0] + (width // 2)),
#             )
#             return s

#         slices = np.apply_along_axis(makeSlice, 2, self.centers, *[(self.stampWidth)])

#         def stampImg(slices, img):
#             return img[tuple(slices)]

#         xdim = self.device.dims.x
#         ydim = self.device.dims.y
#         imgstamps = np.apply_along_axis(stampImg, 2, slices, *[(img)])
#         indices = np.array(
#             [[x, y] for x in range(0, xdim) for y in range(0, ydim)]
#         ).reshape(xdim, ydim, 2)

#         a = np.empty((xdim, ydim), dtype=np.object)
#         for x, y in indices.reshape((xdim * ydim, 2)):
#             stamp = imgstamps[x, y]
#             center = self.centers[x, y]
#             s = tuple(slices[x, y])
#             a[x, y] = Stamp(
#                 stamp,
#                 center,
#                 s,
#                 (x + 1, y + 1),
#                 self.pinlist.loc[x + 1, y + 1][pinlistFeatureTitle],
#             )
#         return a

#     @staticmethod
#     def quadrilateralInterp(corners, dims):
#         """
#         Grid the chip using the corners as vertices and the dims as the number of latice points
#         in the x and y directions.

#         Arguments:
#             (tuple | namedtuple) corners: cornerpositions of the form
#                 ((ULx, ULy),(URx, URy),(LLx, LLy),(LRx, LRy))
#             (tuple) dims: chip dimensions (num columns, num rows)

#         Returns:
#             (np.ndarray) a 2-D array of latice (x, y) coordinates
#         """

#         def interp(p1, p2, divs):
#             y = np.linspace(p1[0], p2[0], divs, dtype=int)
#             x = np.linspace(p1[1], p2[1], divs, dtype=int)
#             return np.stack((x, y), axis=1)

#         if not isinstance(corners, type(namedtuple)):
#             chipCorners = namedtuple("Corners", ["ul", "ur", "bl", "br"])
#             corners = chipCorners(*corners)

#         left = interp(corners.ul, corners.bl, dims.y)
#         right = interp(corners.ur, corners.br, dims.y)
#         mesh = [interp(p1, p2, dims.x) for p1, p2 in zip(left, right)]
#         return np.stack(mesh, axis=1)

#     def mapto(self, target, features="all"):
#         """
#         Maps the chamber and/or button parameters to the target ChipImage, and generates the
#         Chamber and/or Button objects for those features.

#         Arguments:
#             (ChipImage) target:
#             (str) features; features to map ('chamber', 'button', 'all')

#         Returns:
#             None

#         """

#         dims = self.device.dims
#         indices = [(i, j) for i in range(dims.x) for j in range(dims.y)]
#         for i in indices:
#             x, y = i
#             s = self.stamps[x, y]
#             t = target.stamps[x, y]
#             if features == "chamber":
#                 t.defineChamber(s.chamber.center, s.chamber.radius)
#             elif features == "button":
#                 t.defineButton(
#                     s.button.center, s.button.disk_radius, s.button.annulus_radii
#                 )
#             elif features == "all":
#                 t.defineChamber(s.chamber.center, s.chamber.radius)
#                 t.defineButton(
#                     s.button.center, s.button.disk_radius, s.button.annulus_radii
#                 )
#             else:
#                 raise ValueError(
#                     'Invalid feature name. Choices are "chamber", "button", or "all".'
#                 )

#     def findChambers(self, coerce_center=False):
#         """
#         Performs chamber finding for each of the Stamps in the ChipImage. Uses a Hough transform.

#         Arguments:
#             None

#         Returns:
#             None

#         """

#         for c in self.stamps.flatten():
#             c.findChamber(coerce_center=coerce_center)

#     def findButtons(self):
#         """
#         Performs button finding for each of the Stamps in the ChipImage. Uses a Hough transform.

#         Arguments:
#             None

#         Returns:
#             None

#         """

#         for c in tqdm(self.stamps.flatten(), desc="Finding Buttons"):
#             c.findButton()

#     def summarize(self):
#         """
#         Summarizes the ChipImage feature parameters and returns a Pandas DataFrame of the result.

#         Arguments:
#             None

#         Returns:
#             (pd.DataFrame) a Pandas DataFrame summarizing the ChipImage feature parameters

#         """

#         records = {}
#         for s in self.stamps.flatten():
#             records[s.index] = s.summarize()
#         s = pd.DataFrame.from_dict(records, orient="index").sort_index()
#         s.index.rename(["x", "y"], inplace=True)
#         return s

#     def _delete_stamps(self):
#         """
#         Deletes and forces garbage collection on the image data contained in the ChipImage stamps.
#         Data include the stamp data, the chamber and/or button stamp data, and the feature masks.

#         Arguments:
#             None

#         Returns:
#             None

#         """

#         for s in self.stamps.flatten():
#             del s.data
#             s.data = None
#             if s.chamber:
#                 del s.chamber.stampdata
#                 del s.chamber.disk
#                 s.chamber.disk = None
#                 s.chamber.stampdata = None
#             if s.button:
#                 del s.button.stampdata
#                 del s.chamber.disk
#                 del s.chamber.annulus
#                 s.chamber.disk = None
#                 s.chamber.annulus = None
#                 s.button.stampdata = None
#         gc.collect()

#     def summary_image(self, stamptype):
#         """
#         Generates a "deflated" chip image as a numpy ndarray. Returns the
#         the ChipImage stamps concatenated into a single array (image)

#         Arguments:
#             (str) stamptype: parameterized feature type to draw onto stamp ('chamber' | 'button')

#         Returns:
#             (np.ndarray) a 2-D numpy ndarray of the concatenated stamps

#         """

#         return ChipImage.stitch2D(self._summary_image_arr(stamptype))

#     def _summary_image_arr(self, stamptype):
#         """
#         Generates a 2-d numpy array of stamp images (2-d numpy arrays), indexed by their chip indices.

#         Arguments:
#             (str) stamptype: parameterized feature type to draw onto stamp ('chamber' | 'button')

#         Returns:
#             (np.ndarray) a 2-D numpy array of stamp data (ndarrays)
#         """

#         tiles = self.stamps.flatten()
#         stampdims = tiles[0].summary_stamp(stamptype).shape

#         arrShape = [
#             self.device.dims[0],
#             self.device.dims[1],
#             stampdims[0],
#             stampdims[1],
#         ]
#         r = np.array([s.summary_stamp(stamptype) for s in tiles]).reshape(*arrShape)
#         return r

#     @staticmethod
#     def stitch2D(arr):
#         """
#         Concatenates a 2-d array of 2-d numpy arrays (nested) into a single 2-d array (concatenates
#         in both directions)

#         Arguments:
#             (np.ndarray) arr: nested ndarray of the form[[arr1.1, arr1.1,...], [arr2.1, arr2.2,...], ...]

#         Returns:
#             (np.ndarray) concatenated 2-d array

#         """

#         rowsStitched = np.concatenate(arr, axis=2)  # Stitch rows
#         fullStitched = np.concatenate(rowsStitched, axis=0)  # Stitch cols
#         return fullStitched

#     def repo_dump(self, stamptype, target_root, title, as_ubyte=False):
#         """
#         Saves the stamps of the ChipImage to the target directory as a stamp repository (repo).

#         Arguments:
#             (str) stamptype:parameterized feature type to draw onto stamp ('chamber' | 'button')
#             (str) target_root: repo directory, or directory in which to instantiate repo
#             (str) title: stem of filename
#             (bool) as_ubyte: flag to save stamp as uint8 image

#         Returns:
#             None

#         """
#         # saves each stamp to a repo of the form root->id->index
#         import os
#         from pathlib import Path

#         for stamp in self.stamps.flatten():
#             sid = stamp.id
#             index = "{}_{}".format(*stamp.index)
#             s = stamp.summary_stamp(stamptype)  # uint8 for export (space saving)
#             if as_ubyte:
#                 s = skimage.img_as_ubyte(s)
#             target = Path(os.path.join(target_root, sid, index, "{}.png".format(title)))
#             os.makedirs(target.parent, exist_ok=True)
#             skimage.io.imsave(target, s)

#     # def __str__(self):
#     #     return "IDs: {}, Device: {}, ImageReference: {}".format(
#     #         self.ids, str((self.device.setup, self.device.dname)), self.data_ref
#     #     )


# class Stamp:
#     # chamberrad = 16
#     chamberrad = 32
#     outerchamberbound = 5
#     circlePara1Index = 50
#     circlePara2Index = 40

#     def __init__(self, img, center, slice, index, id):
#         """
#         Constructor for a Stamp object, which contains feature data and parameters and permits
#         feature finding.

#         Arguments:
#             (np.ndarray) img:
#             (tuple) center
#             (tuple) slice
#             (tuple) index
#             (str | int) id:

#         Returns:
#             None

#         """

#         self.data = img  # the actual stamp data
#         self.index = index
#         self.slice = slice
#         self.id = id
#         self.chamber = None
#         self.button = None

#     def defineChamber(self, center, radius):
#         """
#         Manually defines a chamber. The passed center coordinates are with respect to the stamp
#         coordinate system (origin upper left).

#         Arguments:
#             (tuple) center:
#             (int) radius:

#         Returns:
#             None

#         """
#         if center != center:
#             self.chamber = Chamber.BlankChamber()
#         else:
#             p = Stamp.circularSubsection(self.data, center, radius)
#             self.chamber = Chamber(self.data, p["mask"], p["center"], p["radius"])

#     def defineButton(self, center, radius, annulus_radii):
#         """
#         Manually defines a Button. The passed center coordinates are with respect to the stamp
#         coordinate system (origin upper left).

#         Arguments:
#             (tuple) center:
#             (int) radius:
#             (tuple) annulus radii:

#         Returns:
#             None

#         """

#         b = Stamp.circularSubsection(self.data, center, radius)
#         o = Stamp.circularSubsection(
#             self.data, center, annulus_radii[1]
#         )  # The circles can extend past the edge of the image

#         b_mask = b["mask"]
#         o_mask = o["mask"]
#         annulus_mask = ~(o_mask ^ b_mask)
#         self.button = Button(
#             self.data,
#             b_mask,
#             annulus_mask,
#             b["center"],
#             b["radius"],
#             (b["radius"], b["radius"]),
#         )

#     def summarize(self):
#         """
#         Summarizes a stamp as a dictionary of parameterized chamber, button, and stamp features.

#         Arguments:
#             None

#         Returns:
#             (dict) stamp features

#         """

#         c_r = {}
#         b_r = {}
#         if self.chamber:
#             c_r = self.chamber.summary
#         if self.button:
#             b_r = self.button.summary
#         stampInfo = {
#             "xslice": (self.slice[0].start, self.slice[0].stop),
#             "yslice": (self.slice[1].start, self.slice[1].stop),
#             "id": self.id,
#         }
#         return {**c_r, **b_r, **stampInfo}

#     def summary_stamp(self, stamptype):
#         """
#         Annotes a stamp image and overlays chamber or button borders.

#         Arguments:
#             (str) stamptype: parameterized feature type to draw onto stamp ('chamber' | 'button')

#         Returns:
#             (np.ndarray) the annotated stamp image array

#         """
#         # stamptype: ('chamber', 'button')
#         if stamptype == "chamber":
#             circles = [[self.chamber.radius, self.chamber.center]]
#             index = "{}.{} | {}".format(self.index[0], self.index[1], self.id)
#             return annotateStamp(self.data, circles, index, "")
#         elif stamptype == "button":
#             circles = [
#                 [self.button.disk_radius, self.button.center],
#                 [self.button.annulus_radii[1], self.button.center],
#             ]
#             index = "{}.{} | {}".format(self.index[0], self.index[1], self.id)
#             val = "{}, {}".format(
#                 int(self.button.summary["summed_button_BGsub"]),
#                 int(self.button.summary["summed_button_annulus_normed"]),
#             )
#             return annotateStamp(self.data, circles, index, val)
#         else:
#             raise ValueError(
#                 'Invalid stamp type. Valid values are "chamber" or "button"'
#             )

#     @staticmethod
#     def circularSubsection(img, center, radius):
#         """
#         Given an image stamp, chamber/button center position and radius, returns the raw and
#         analyzed chamber/button pixel values for that chamber/button. Uses a mask.

#         Arguments:
#             (np.ndarray) image:
#             (tuple) center: x,y center location of the image
#             (int) radius: radius of the chamber/button to be masked

#         Returns:
#             (dict) circularSubsection features

#         """

#         imageCopy = img.copy()
#         mask = np.zeros(imageCopy.shape)
#         cv2.circle(mask, center, radius, 1, -1)  # Warning: MODIFIES mask IN PLACE!!
#         mask = mask.astype(np.bool)

#         insert = np.where(mask)
#         intensities = img[insert]

#         return {
#             "mask": ~mask,
#             "intensities": intensities,
#             "center": center,
#             "radius": int(radius),
#         }

#     def findChamber(self, coerce_center=False):
#         """
#         Uses Hough transform to find a chamber.

#         Arguments:
#             (np.ndarray) imageCopy: chamber stamp image.

#         Returns:
#             (dict) optimizedSpotParams: optimal found chamber border parameters

#         """

#         chamberRadius = Stamp.chamberrad
#         outerChamberBound = Stamp.outerchamberbound
#         img = self.data

#         if coerce_center:
#             p = Stamp.circularSubsection(
#                 img, (int(img.shape[0] / 2), int(img.shape[0] / 2)), chamberRadius
#             )
#             self.chamber = Chamber(img, p["mask"], p["center"], p["radius"])
#             return

#         else:
#             with warnings.catch_warnings():
#                 warnings.simplefilter(
#                     "ignore"
#                 )  # Will throw warning due to precision loss
#                 cimg = skimage.img_as_ubyte(img, force_copy=True)

#             # searchRadii
#             minRad = chamberRadius
#             maxRad = minRad + outerChamberBound

#             # find circles
#             circles = cv2.HoughCircles(
#                 cimg,
#                 cv2.HOUGH_GRADIENT,
#                 2,
#                 10,
#                 param1=self.circlePara1Index,
#                 param2=self.circlePara2Index,
#                 minRadius=minRad,
#                 maxRadius=maxRad,
#             )

#             circlePara1Index = self.circlePara1Index
#             # If no circles found, loosen gradient threshold
#             while type(circles) is not np.ndarray and circlePara1Index > 5:
#                 circles = cv2.HoughCircles(
#                     cimg,
#                     cv2.HOUGH_GRADIENT,
#                     2,
#                     10,
#                     param1=circlePara1Index,
#                     param2=self.circlePara2Index,
#                     minRadius=minRad + 1,
#                     maxRadius=maxRad + 2,
#                 )
#                 circlePara1Index -= 1

#             # If still none found, return a blank chamber (failed)
#             if not np.any(circles):
#                 m = "No chamber border found for chamber {}".format(str(self.index))
#                 warnings.warn(m)
#                 self.chamber = Chamber.BlankChamber()
#                 return

#             # Else, round the resulting circle params
#             circles = np.around(circles)

#             # Then select circle of highest summed I from those found
#             bestCircle = None
#             bestIntensity = 0
#             for i in circles[0, :]:
#                 if len(circles[0, :]) == 1:
#                     bestCircle = i
#                     break
#                 # pick the set of circles that maximizes intensity inside of it (may have ties)
#                 circleResultsSum = np.sum(
#                     Stamp.circularSubsection(cimg, (int(i[0]), int(i[1])), int(i[2]))[
#                         "intensities"
#                     ]
#                 )
#                 if circleResultsSum > bestIntensity:
#                     bestIntensity = circleResultsSum
#                     bestCircle = i

#             # Calculate the final parameters
#             p = Stamp.circularSubsection(
#                 img, (int(bestCircle[0]), int(bestCircle[1])), int(bestCircle[2])
#             )
#             self.chamber = Chamber(img, p["mask"], p["center"], p["radius"])

#     def findButton(self):
#         """
#         Button finding algorithm using Craig's "grid search" optimization.
#         Searches sparse grid of tile position centers, finds optimum, then refines by searching local
#         neighborhood. Then, fits the radius and re-fits the centerposition after each decrease in radius.
#         Terminates when either the minRadius is reached or finds a bright circle with small standard deviation
#         within the found circle border falls below specified threshold.

#         Arguments:
#             None

#         Returns:
#             (dict) bestSpotParams: {'mask': ~mask, 'intensities': intensities, 'center': center, 'radius': int(radius)}

#         """

#         ######## DEFAULTS ########
#         searchSpacing = 7
#         radius = 15  # was 2x2 = 15, 4x4 = 7
#         tileWidth = 110
#         tileHeight = 110
#         refiningRange = 7
#         minRadius = 9
#         stdCutoff = 0.9
#         boundingInsetRatio = 0.3
#         ###########################

#         imagestamp = self.data

#         maxI = 0
#         bestSpotParams = None
#         fitRadius = radius
#         localBGRadius = radius * 2

#         boundingInset = int(tileWidth * boundingInsetRatio)

#         # Crude initial fit of center position (sparse initial serach grid, entire image stamp) by maximizing summed intensity
#         for xIndex in range(boundingInset, tileWidth - boundingInset, searchSpacing):
#             for yIndex in range(
#                 boundingInset, tileHeight - boundingInset, searchSpacing
#             ):
#                 features = Stamp.circularSubsection(
#                     imagestamp, (xIndex, yIndex), fitRadius
#                 )
#                 summedI = np.nansum(features["intensities"])
#                 if summedI > maxI:
#                     maxI = deepcopy(summedI)
#                     bestSpotParams = deepcopy(features)

#         # If the image is perfectly black in the bounding region, it's necessary to just pick the center position as a placeholder
#         if not bestSpotParams:
#             warnmsg = "No intensity observed for chamber {}".format(self.index)
#             warnings.warn(warnmsg)
#             bestSpotParams = deepcopy(
#                 Stamp.circularSubsection(
#                     imagestamp, (int(tileWidth / 2), int(tileHeight / 2)), fitRadius
#                 )
#             )
#             buttonBound = Stamp.circularSubsection(
#                 imagestamp, bestSpotParams["center"], radius
#             )
#             outerBound = Stamp.circularSubsection(
#                 imagestamp, bestSpotParams["center"], localBGRadius
#             )  # The circles can extend past the edge of the image
#             b_mask = buttonBound["mask"]
#             o_mask = outerBound["mask"]
#             annulus_mask = ~(o_mask ^ b_mask)
#             self.button = Button(
#                 imagestamp,
#                 b_mask,
#                 annulus_mask,
#                 buttonBound["center"],
#                 buttonBound["radius"],
#                 (buttonBound["radius"], outerBound["radius"]),
#             )
#             return

#         # Fine-tuning center position (dense local array for search grid) by maximizing summed intensity
#         for xIncrement in np.linspace(
#             bestSpotParams["center"][0] - refiningRange,
#             bestSpotParams["center"][0] + refiningRange,
#             num=2 * refiningRange,
#             dtype=int,
#         ):
#             for yIncrement in np.linspace(
#                 bestSpotParams["center"][1] - refiningRange,
#                 bestSpotParams["center"][1] + refiningRange,
#                 num=2 * refiningRange,
#                 dtype=int,
#             ):
#                 features = Stamp.circularSubsection(
#                     imagestamp, (xIncrement, yIncrement), fitRadius
#                 )
#                 summedI = np.nansum(features["intensities"])
#                 if summedI > maxI:
#                     maxI = deepcopy(summedI)
#                     bestSpotParams = deepcopy(features)

#         refStdDev = np.nanstd(bestSpotParams["intensities"])

#         # Refines center position by optimizing radius via watershed method
#         while bestSpotParams["radius"] > minRadius:
#             fitRadius -= 1

#             maxIAtRadius = 0
#             bestParamsAtRadius = bestSpotParams

#             for xIncrement in np.linspace(
#                 bestParamsAtRadius["center"][0] - refiningRange,
#                 bestParamsAtRadius["center"][0] + refiningRange,
#                 num=2 * refiningRange,
#                 dtype=int,
#             ):
#                 for yIncrement in np.linspace(
#                     bestParamsAtRadius["center"][1] - refiningRange,
#                     bestParamsAtRadius["center"][1] + refiningRange,
#                     num=2 * refiningRange,
#                     dtype=int,
#                 ):
#                     features = Stamp.circularSubsection(
#                         imagestamp, (xIncrement, yIncrement), fitRadius
#                     )
#                     summedI = np.nansum(features["intensities"])
#                     if summedI > maxIAtRadius:
#                         maxIAtRadius = deepcopy(summedI)
#                         bestParamsAtRadius = deepcopy(features)
#             # If the radius has shrunk the optimal circle to w/in bright bounds, stop the fitting and use that circle center)
#             if np.nanstd(bestParamsAtRadius["intensities"]) < stdCutoff * refStdDev:
#                 bestSpotParams = Stamp.circularSubsection(
#                     imagestamp, bestParamsAtRadius["center"], radius
#                 )
#                 break
#             else:
#                 maxI = deepcopy(maxIAtRadius)
#                 bestSpotParams = deepcopy(bestParamsAtRadius)

#         # If radius fitting didn't work, just go with the parameters from before
#         buttonBound = Stamp.circularSubsection(
#             imagestamp, bestSpotParams["center"], radius
#         )
#         outerBound = Stamp.circularSubsection(
#             imagestamp, bestSpotParams["center"], localBGRadius
#         )  # The circles can extend past the edge of the image

#         b_mask = buttonBound["mask"]
#         o_mask = outerBound["mask"]
#         annulus_mask = ~(o_mask ^ b_mask)

#         self.button = Button(
#             imagestamp,
#             b_mask,
#             annulus_mask,
#             buttonBound["center"],
#             buttonBound["radius"],
#             (buttonBound["radius"], outerBound["radius"]),
#         )

#     def __str__(self):
#         return "Stamp| ID:{}, Index:{}".format(self.id, self.index)

#     def _repr_pretty_(self, p, cycle=True):
#         p.text("<{}>".format(self.__str__()))


# class Chamber:
#     def __init__(self, stampdata, disk, center, radius, empty=False):
#         """
#         Constructor for a Chamber object

#         Arguments:
#             (np.ndarray) stampdata: the original stamp data
#             (np.ndarray) disk: a boolean mask for the stampdata FALSE within the found chamber
#             (tuple) center: chamber center coordinates, with respect to stampdata coord. system
#             (int) radius: chamber radius
#             (bool) empty: flag for empty chamber

#         Returns:
#             None

#         """

#         self.blankFlag = empty
#         self.stampdata = stampdata  # uint16 ndarray
#         self.disk = disk  # a mask
#         self.disk_intensities = ma.compressed(ma.array(stampdata, mask=disk))
#         self.center = center
#         self.radius = radius
#         self.summary = self.summarize()

#     def get_disk(self):
#         """
#         Generates the masked array corresponding to the chamber.

#         Arguments:
#             None

#         Returns:
#             (np.ma) masked array of the stamp chamber

#         """

#         return ma.array(self.stampdata, mask=self.disk)

#     def summarize(self):
#         """
#         Summarizes a chamber as a dictionary of paramters mapped from their descriptors

#         Arguments:
#             None

#         Returns:
#             (dict) summary of chamber parameters

#         """

#         features = [
#             "median_chamber",
#             "sum_chamber",
#             "std_chamber",
#             "x_center_chamber",
#             "y_center_chamber",
#             "radius_chamber",
#         ]

#         if self.blankFlag:
#             return dict(zip(features, list(np.full(len(features), np.nan))))

#         disk = self.get_disk()
#         medI = int(ma.median(disk))
#         sumI = int(disk.sum())
#         sdI = int(disk.std())

#         vals = [medI, sumI, sdI, self.center[0], self.center[1], self.radius]
#         return dict(zip(features, vals))

#     @classmethod
#     def BlankChamber(cls):
#         """
#         Factory fresh empty chamber, hot off the press
#         """
#         bc = cls(*([np.nan] * 4), **{"empty": True})
#         return bc


# class Button:
#     localBG_Margin = 10

#     def __init__(
#         self, stampdata, disk, annulus, center, disk_radius, annulus_radii, empty=False
#     ):
#         """
#         Constructor for a Button object

#         Arguments:
#             (np.ndarray) stampdata: the original stamp data
#             (np.ndarray) disk: a boolean mask for the stampdata FALSE within the found chamber
#             (np.ndarray) annulus: a boolean mask for the stampdata FALSE within the button annulus
#                 (local background)
#             (tuple) center: chamber center coordinates, with respect to stampdata coord. system
#             (int) disk_radius: button radius
#             (tuple) annulus radii: inner and outer radii of the annulus (innerrad, outerrad)
#             (bool) empty: flag for empty button

#         Returns:
#             None

#         """

#         self.blankFlag = empty
#         self.stampdata = stampdata  # uint16 ndarray
#         self.disk = disk  # a mask
#         self.disk_intensities = ma.compressed(self.get_disk())
#         self.annulus = annulus  # a mask
#         self.annulus_intensities = ma.compressed(self.get_annulus())
#         try:
#             self.annulus_to_disk_ratio = len(self.annulus_intensities) / len(
#                 self.disk_intensities
#             )
#         except:
#             warnings.warn(
#                 "Annulus ratio could not be calculated.\nButton Intensities Are Of Length Zero.\
#                             Annulus to disk ratio is NaN"
#             )
#             self.annulus_to_disk_ratio = np.nan
#         self.center = center
#         self.disk_radius = disk_radius
#         self.annulus_radii = annulus_radii
#         self.summary = self.summarize()

#     def get_disk(self):
#         """
#         Generates the masked array corresponding to the button.

#         Arguments:
#             None

#         Returns:
#             (np.ma) masked array of the stamp button

#         """

#         return ma.array(self.stampdata, mask=self.disk)

#     def get_annulus(self):
#         """
#         Generates the masked array corresponding to the annulus.

#         Arguments:
#             None

#         Returns:
#             (np.ma) masked array of the stamp button annulus

#         """

#         return ma.array(self.stampdata, mask=self.annulus)

#     def summarize(self):
#         """
#         Summarizes a button as a dictionary of paramters mapped from their descriptors

#         Arguments:
#             None

#         Returns:
#             (dict) summary of chamber parameters

#         """

#         features_disk = [
#             "median_button",
#             "summed_button",
#             "summed_button_BGsub",
#             "std_button",
#             "mean_button",
#             "x_button_center",
#             "y_button_center",
#             "radius_button_disk",
#         ]
#         features_ann = [
#             "median_button_annulus",
#             "summed_button_annulus_normed",
#             "std_button_annulus_localBG",
#             "mean_annulus",
#             "inner_radius_button_annulus",
#             "outer_radius_button_annulus",
#         ]

#         if self.blankFlag:
#             return dict(
#                 zip(
#                     features_disk + features_ann,
#                     list(np.full(len(features_disk + features_ann), np.nan)),
#                 )
#             )

#         disk = self.get_disk()
#         annulus = self.get_annulus()
#         medI_disk = int(ma.median(disk))
#         sumI_disk = int(disk.sum())
#         sdI_disk = int(disk.std())
#         meanI_disk = int(disk.mean())
#         medI_ann = int(ma.median(annulus))
#         sumI_ann_normed = int(annulus.sum() / self.annulus_to_disk_ratio)
#         sdI_ann = int(annulus.std())
#         meanI_ann = int(annulus.mean())
#         sumI_BGsub = sumI_disk - sumI_ann_normed

#         vals_disk = [
#             medI_disk,
#             sumI_disk,
#             sumI_BGsub,
#             sdI_disk,
#             meanI_disk,
#             self.center[0],
#             self.center[1],
#             self.disk_radius,
#         ]
#         vals_ann = [
#             medI_ann,
#             sumI_ann_normed,
#             sdI_ann,
#             meanI_ann,
#             self.annulus_radii[0],
#             self.annulus_radii[1],
#         ]
#         return dict(zip(features_disk + features_ann, vals_disk + vals_ann))

#     @classmethod
#     def BlankButton(cls):
#         bb = cls(*([np.nan] * 6), **{"empty": True})
#         return bb


# def annotateStamp(data, circles, index, val):
#     """
#     Annotates a stamp image with an index, a feature value, and arbitrary circles
#     (for chamber and button feature drawing). Text is drawn in white. Circles are drawn in
#     black and white

#     The index a(nd value) is/are drawn on the top of the stamp.

#     Arguments:
#         (tuple) circle paramters of the form (radius, (centerx, centery))
#         (str) index: stamp index
#         (str) val: stamp value

#     Returns:
#         (np.ndarray) annotated stamp image

#     """

#     working = deepcopy(data)
#     d = cv2.copyMakeBorder(deepcopy(data), 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=60000)
#     cv2.putText(d, index, (2, 12), cv2.FONT_HERSHEY_PLAIN, 0.8, 60000)  # Index
#     cv2.putText(d, val, (2, len(d) - 4), cv2.FONT_HERSHEY_PLAIN, 0.7, 60000)  # Index
#     for rad, center in circles:
#         if center != center:
#             pass
#         else:
#             cv2.circle(d, center, rad + 2, 0, thickness=1)
#             cv2.circle(d, center, rad + 1, (2**16) - 1, thickness=1)
#             cv2.circle(d, center, rad, 0, thickness=1)
#     return d

In [ ]:
# from htbam_analysis.stitching import rastering
# from htbam_analysis.stitching.stitching import stitch_single_raster

# import shutil

# class ImageStitcher:

#     def __init__(
#             self,
#             root_path,
#             ):

#         self.root_path = root_path
#         self._raw_images_path = root_path / 'raw_images'
#         self._stitched_images_path = root_path / 'stitched_images'

#         self.raster_data = self.load_raster_data()
#         self.stitched_image_data = None
        
#     def load_raster_data(self):

#         raster_data = pd.read_csv(self.root_path / 'imaging.csv')
    
#         # make absolute image paths
#         raster_data['image_path'] = raster_data['image_path'].apply(Path)
#         raster_data['image_path'] = raster_data['image_path'].apply(lambda p: self._raw_images_path / p)
#         raster_data['image_path_parent'] = raster_data['image_path'].apply(lambda p: p.parent)
#         raster_data.sort_values(by=['image_path_parent', 'raster_col_index', 'raster_row_index'], inplace=True)

#         return raster_data

#     def stitch_images(self, rotation: float, acqui_origin: Tuple[bool] = (True, False)):

#         assert self.raster_data, 'Raster data has not been set.'

#         # TODO: make this not hard-coded
#         SIZE = 1600
#         raster_headers = ['image_path', 'image_path_parent', 'raster_index', 'x', 'y', 'z', 'frame_time', 'raster_width', 'raster_height', 'raster_overlap', 'raster_row_index', 'raster_col_index']
#         stitched_image_data = []

#         grouped = self.raster_data.groupby('image_path_parent')

#         for image_parent, df in grouped:

#             # format outpath and make sure directory to write it to exists
#             outpath = self._stitched_images_path / image_parent.relative_to(self._raw_images_path).with_suffix('.tif')
#             outpath.parent.mkdir(exist_ok=True)

#             # extract params, stitch image
#             overlap, width, height = df[['raster_overlap', 'raster_width', 'raster_height']].iloc[0]
#             raster = df['image_path'].to_list()
#             params = rastering.RasterSettings(overlap, size=SIZE, acqui_ori=acqui_origin, rotation=rotation, dims=(width, height), auto_ff=True, ff_type='BaSiC') 
#             stitch_single_raster(raster, params, None)

#             # carry over metadata
#             row = df.drop_duplicates(subset=['image_path_parent']).drop(columns=raster_headers)
#             row['image_path'] = outpath.relative_to(self._stitched_images_path)
#             stitched_image_data.append(row)

#         self.stitched_image_data = pd.concat(stitched_image_data).reset_index(drop=True)
#         self.stitched_image_data.to_csv(self._stitched_images_path / 'stitched_images.csv')


# def backgroud_subtract(background_image, target_image):

#     # TODO: figure out why on earth this is hard-coded
#     MAX_VALUE = 65535

#     subtracted = np.subtract(target_image.astype("float"), background_image.astype("float"))
#     subtracted_clipped = np.clip(subtracted, 0, MAX_VALUE).astype("uint16")
    
#     return subtracted_clipped


# class BackgroundSubtractor:

#     def __init__(self, root_path):

#         self.root_path = root_path
#         self._stitched_image_path = root_path / 'stitched_images'
#         self._bgsub_images_path = root_path / 'bgsub_images'

#         self.stitched_image_data = self.load_stitched_image_data()
#         self.bgsub_data = None

#     def load_stitched_image_data(self):

#         stitched_image_data = pd.read_csv(self._stitched_image_path / 'stitched_images.csv')

#         # make absolute image paths
#         stitched_image_data['image_path'] = stitched_image_data['image_path'].apply(Path)
#         stitched_image_data['image_path'] = stitched_image_data['image_path'].apply(lambda p: self._stitched_image_path / p)

#         return stitched_image_data
    
#     def _background_image_check(self, background_images: List[Union[Path, str]]):

#         reformatted = []
#         for image in background_images:
#             image = Path(image) if not isinstance(image, Path) else image
#             assert image.exists(), 'Background image not found.'

#             mask = self.stitched_image_data['image_path'] == image
#             n_images = mask.sum()

#             if n_images < 1:
#                 raise Exception('{} not found in `stitched_images.csv`.'.format(image))
            
#             if n_images > 1:
#                 raise Exception('Duplicate entries for {} found in `stitched_images.csv`.'.format(image))
            
#             reformatted.append(image)
            
#         return reformatted
    
#     def subtract(
#             self, 
#             background_images: List[Union[Path, str]],
#             settings_to_match: List[str] = ['temp', 'hum','setup', 'dname', 'lightsource', 'channel', 'exposure', 'camera_mode', 'binning', 'nosepiece'],
#             verbose: bool = True
#             ):

#         # input sanity check
#         background_images = self._background_image_check(background_images)

#         bgsub_data = []
#         grouped = self.stitched_image_data.groupby(by=settings_to_match, dropna=False)

#         for settings, group in grouped:

#             # region background image sanity checks

#             print('Attempting background subtracting on images with the following settings:')
#             print(' | '.join(['{}: {}'.format(setting, value) for setting, value in zip(settings_to_match, settings)]))

#             # make a target df
#             background_image = None
#             group['background_image'] = [background_image] * len(group)

#             # check if a unique background image can be found
#             background_mask = group['image_path'].isin(background_images)
    
#             if sum(background_mask) == 0:
#                 print('No background image found. Subtracting with zero array.')

#             else:
#                 # grab background image (should just be one row)
#                 background_image_path = group[background_mask].iloc[0]['image_path']
#                 background_image = io.imread(background_image_path)
#                 print('Using {} as background image'.format(background_image_path))

#                 # update dataframe
#                 group = group.copy() # to stop pandas from complaining 
#                 group['background_image'] = [background_image_path] * len(group)
#                 group = group[~background_mask]

#             # endregion

#             success_mask = np.ones(len(group), dtype=bool)
#             for i in tqdm(range(len(group)), desc='Running background subtraction.'):

#                 target_image_path = group['image_path'].iloc[i]
#                 outfile = self._bgsub_images_path / target_image_path.relative_to(self._stitched_image_path)
#                 outfile.parent.mkdir(parents=True, exist_ok=True)

#                 if isinstance(background_image, np.ndarray):
#                     try: 
#                         # subtract and save
#                         target_image = io.imread(target_image_path)
#                         subtracted_image = backgroud_subtract(background_image, target_image)
#                         io.imsave(outfile, subtracted_image, plugin="tifffile", check_contrast=False)

#                     except Exception as e:
#                         # "subtract" a zero array
#                         # or in other words, just copy target image
#                         success_mask[i] = False
#                         shutil.copy(target_image_path, outfile)
#                         if verbose:
#                             print(f"Error background subtracting {target_image_path}: {e}")
                        
#                 else:
#                     # "subtract" a zero array
#                     # or in other words, just copy target image
#                     success_mask[i] = False
#                     shutil.copy(target_image_path, outfile)   
                             
#             print("{successes} / {total} images were successfully background subtracted".format(successes=(success_mask).sum(), total=len(success_mask)))
#             print()
#             # update failure rows
#             group.loc[~success_mask, 'column_name'] = None
#             bgsub_data.append(group)

#         # format and save the background subtraction dataframe
#         self.bgsub_data = pd.concat(bgsub_data, ignore_index=True)
#         self.bgsub_data['image_path'] = self.bgsub_data['image_path'].apply(lambda f: f.relative_to(self._stitched_image_path))
#         self.bgsub_data['background_image'] = self.bgsub_data['background_image'].apply(lambda f: None if not f else f.relative_to(self._stitched_image_path))
#         self.bgsub_data.to_csv(self._bgsub_images_path / 'bgsub_images.csv', index=False)



In [ ]:
root = Path('/Volumes/JSZ2/deez')

stitcher = ImageStitcher(root)
stitcher.stitch_images(rotation=-2, acqui_origin=(True, False))

# paths to all background images
# use full, absolute paths
background_images = [
    '/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-22-50_d1_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif',
    '/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-22-50_d1_aura_UV340_5_1000_dynamic_range_2x2_4_nadh_background.tif',
    '/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-23-58_d2_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif',
    '/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-23-58_d2_aura_UV340_5_1000_dynamic_range_2x2_4_nadh_background.tif'
]
settings_to_match = ['temp', 'hum','setup', 'dname', 'lightsource', 'channel', 'exposure', 'camera_mode', 'binning', 'nosepiece']
subtractor = BackgroundSubtractor(root)
subtractor.subtract(
    background_images=background_images,
    settings_to_match=settings_to_match
)

/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-22-50_d1_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif
/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-22-50_d1_aura_UV340_5_1000_dynamic_range_2x2_4_nadh_background.tif
/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-23-58_d2_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif
/Volumes/JSZ2/deez/stitched_images/2026-03-05_12-23-58_d2_aura_UV340_5_1000_dynamic_range_2x2_4_nadh_background.tif
Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d1 | lightsource: aura_UV340 | channel: 5 | exposure: 500 | camera_mode: dynamic_range | binning: 2x2 | nosepiece: 4
Using /Volumes/JSZ2/deez/stitched_images/2026-03-05_12-22-50_d1_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif as background image


Running background subtraction.: 100%|██████████| 15/15 [00:32<00:00,  2.15s/it]


15 / 15 images were successfully background subtracted

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d1 | lightsource: aura_UV340 | channel: 5 | exposure: 1000 | camera_mode: dynamic_range | binning: 2x2 | nosepiece: 4
Using /Volumes/JSZ2/deez/stitched_images/2026-03-05_12-22-50_d1_aura_UV340_5_1000_dynamic_range_2x2_4_nadh_background.tif as background image


Running background subtraction.: 100%|██████████| 14/14 [00:35<00:00,  2.55s/it]


14 / 14 images were successfully background subtracted

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d2 | lightsource: aura_UV340 | channel: 5 | exposure: 500 | camera_mode: dynamic_range | binning: 2x2 | nosepiece: 4
Using /Volumes/JSZ2/deez/stitched_images/2026-03-05_12-23-58_d2_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif as background image


Running background subtraction.: 100%|██████████| 15/15 [00:34<00:00,  2.32s/it]


15 / 15 images were successfully background subtracted

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d2 | lightsource: aura_UV340 | channel: 5 | exposure: 1000 | camera_mode: dynamic_range | binning: 2x2 | nosepiece: 4
Using /Volumes/JSZ2/deez/stitched_images/2026-03-05_12-23-58_d2_aura_UV340_5_1000_dynamic_range_2x2_4_nadh_background.tif as background image


Running background subtraction.: 100%|██████████| 14/14 [00:26<00:00,  1.88s/it]

14 / 14 images were successfully background subtracted



In [99]:
def read_pinlist(pl):
    pl["Indices"] = pl.Indices.apply(eval)
    pl["x"] = pl.Indices.apply(lambda x: x[0])
    pl["y"] = pl.Indices.apply(lambda x: x[1])
    sorted_pinlist = pl.set_index(["x", "y"], drop=True, inplace=False).sort_index()
    return sorted_pinlist


def make_dummy_pinlist(block_descriptions: dict):

    def get_block(c):
        return ((c // 8) + 1)

    pinlist_dict = []
    for c in range(32):
        for r in range(56):
            block = get_block(c)
            mutant = block_descriptions[block]
            pinlist_dict.append({'Indices': (c + 1, r + 1), 'MutantID': mutant})
    pinlist = pd.DataFrame(pinlist_dict)
    pinlist['Indices'] = pinlist['Indices'].astype(str)

    pinlist = read_pinlist(pinlist)

    return pinlist


class Device:

    def __init__(self, setup: str, dname: str, dims: Tuple[int, int] = (32, 56)):
        self.setup = setup
        self.dname = dname
        self.dims = namedtuple("ChipDims", ["x", "y"])(*dims)
        self.pinlist = None

    def __str__(self):
        return "{}, {}, {}".format(self.setup, self.dname, self.dims)

    def set_pinlist(self, pinlist_path: Union[str, Path] = None, block_descriptions: dict = None):

        if pinlist_path:
            pinlist = read_pinlist(pd.read_csv(pinlist_path))

        if block_descriptions:
            pinlist = make_dummy_pinlist(block_descriptions)

        self.pinlist = pinlist

        return


class Experiment:

    def __init__(self, root: Union[str, Path]):

        root = Path(root) if not isinstance(root, Path) else root
        self.root = root
        self.devices = {}

        self.image_df = self._load_imaging_data()
        self.stitched_image_df = None
        

    def _load_imaging_data(self):
        
        path = self.root / 'imaging.csv'
        assert path.exists()

        image_df = pd.read_csv(path)
        image_df.sort_values(by=['image_path_parent', 'raster_col_index', 'raster_row_index'], inplace=True)

        return image_df
    

    def stitch_images(self, rotation: float, acqui_ori: Tuple[bool]):

        # sort image df by parent dir and raster position

        # get a list of rasters
        _grouped = self.image_df.groupby('image_path_parent')
        image_dirs = list(_grouped.groups.keys())
        rasters = _grouped['image_path'].apply(list).tolist()

        # make a df for stitched images
        self.stitched_image_df = _grouped.first().reset_index()

        # make raster settings
        raster_settings_list = [rastering.RasterParams(
            overlap=o,
            size=1600,
            acqui_ori = acqui_ori,
            rotation=rotation,
            dims = (w, h),
            auto_ff=True,
            ff_type='BaSiC'
        )
        for o, w, h in zip(self.stitched_image_df['raster_overlap'], self.stitched_image_df['raster_width'], self.stitched_image_df['raster_height'])
        ]

        # drop uneeded columns
        self.stitched_image_df.reset_index(drop=True)
        self.stitched_image_df.drop(columns=['raster_index', 'frame_time', 'raster_width', 'raster_height', 'raster_start_time', 'x', 'y', 'z', 'raster_overlap', 'raster_row_index', 'raster_col_index'], inplace=True)

        # make an output directory for stitched images
        stitched_images = root / 'stitched_images'
        stitched_images.mkdir(exist_ok=True)


    def add_device(self, device: Device):
        self.devices[device.dname] = device


def series_to_dataframe(series_dict):
    """
    Recursively flattens a nested series dictionary into a Pandas DataFrame.
    """
    rows = []

    def walk(item):
        if item.get("item_type") == "image":
            # Create a flat dictionary: {'id': 'img_01', 'metadat_col1': val, ...}
            row = {"identifier": item["identifier"]}
            row.update(item.get("metadata", {}))
            rows.append(row)
        
        elif item.get("item_type") == "series":
            # If it's a series, recurse into its items
            for sub_item in item.get("items", []):
                walk(sub_item)

    # Start the recursion
    walk(series_dict)
    
    return pd.DataFrame(rows)


def df_to_nested_series(df, group_cols, series_id="root"):
    """
    Converts a flat DataFrame into a nested dictionary structure.
    
    Args:
        df: The pandas DataFrame.
        group_cols: A list of column names to group by (e.g., ['replicate', 'channel']).
        series_id: The ID for the current series level.
    """
    # Base Case: No more groups to create, these are individual images
    if not group_cols:
        items = []
        for _, row in df.iterrows():
            # Separate the 'id' from the rest of the metadata
            img_id = str(row['id'])
            # Convert row to dict and remove 'id' so it only contains metadata
            metadata = row.drop('id').to_dict()
            
            items.append({
                "id": img_id,
                "type": "image",
                "metadata": metadata
            })
        return items

    # Recursive Case: Group by the first column in the list
    current_group_col = group_cols[0]
    remaining_cols = group_cols[1:]
    
    nested_items = []
    for group_val, group_df in df.groupby(current_group_col):
        # The ID for the sub-series is the value of the grouping column
        sub_series_id = str(group_val)
        
        nested_items.append({
            "id": sub_series_id,
            "type": "series",
            "items": df_to_nested_series(group_df, remaining_cols, sub_series_id)
        })
        
    # If this is the top-level call, wrap it in a root series dict
    if series_id == "root":
        return {"id": "root", "type": "series", "items": nested_items}
    
    return nested_items


class DataHandler:
    
    def __init__(
            self, 
            root: Union[str, Path]):

        self._root = Path(root) if isinstance(root, str) else root
        assert self._root.exists()
        self._bsgub_images = root / 'bgsub_images'
        assert self._root.exists()

        self.image_metadata = self.load_image_metadata(self._bsgub_images / 'bgsub_images.csv')
        self.series_index_metadata = self.load_series_metadata(self._root / 'series_index.json')

        print("Loaded the following image sets:")
        for key in self.series_index_metadata:
            print(key)

    def load_image_metadata(self, image_metadata_path: Union[str, Path]):
        image_metadata =  pd.read_csv(image_metadata_path)
        image_metadata['image_path'] = image_metadata['image_path'].apply(lambda f: self._bsgub_images / Path(f))
        return image_metadata

    def load_series_metadata(self, series_metadata_path: Union[str, Path]):

        with open(series_metadata_path, 'r') as f:
            series_index = json.load(f)

        series_metadata = {}
        for series in series_index:
            identifier = series['identifier']
            df = series_to_dataframe(series)
            df['identifier'] = df['identifier'].apply(lambda f: self._bsgub_images / Path(f).with_suffix('.tif'))
            series_metadata[identifier] = df
            
        return series_metadata

    def get_images(self, identifier: str):

        series_df = self.series_index_metadata[identifier].copy()
        series_df.sort_values(by='identifier', inplace=True)
        series_mask = self.image_metadata['image_path'].isin(series_df['identifier'])
        
        image_df = self.image_metadata[series_mask].copy()
        
        image_df.sort_values(by='image_path', inplace=True)

        merged = pd.merge(series_df, image_df, left_on='identifier', right_on='image_path')

        return merged


In [91]:
# initialize experiment object
experiment = Experiment(root)

# initialize device(s)
# NOTE: dname NEEDS to exactly match that used within the experiment
d1 = Device(setup='s1', dname='d1', dims=(32, 56))
d1.set_pinlist(block_descriptions={1: 'BLANK', 2: 'BLANK', 3: 'HRAS', 4: 'HRAS'})

d2 = Device(setup='s1', dname='d2', dims=(32, 56))
d2.set_pinlist(block_descriptions={1: 'BLANK', 2: 'BLANK', 3: 'KRAS', 4: 'KRAS'})

# add devices to experiment
experiment.add_device(d1)
experiment.add_device(d2)

- The `DataHandler` class is responsible for managing our two classes of metadata contained with `imaging.csv` and `series_index.json`
- Enables users to select all images associated with a particular assay with ease

In [100]:
data_handler = DataHandler(root)

Loaded the following image sets:
2026-03-05_12-26-07_standard_curve_NADH
2026-03-05_13-17-36_kinetic_series_cycling


In [165]:
class Processor:
    
    def __init__(self, experiment: Experiment, image_data: pd.DataFrame, features: str):
        self.experiment = experiment
        self.image_data = image_data

        self.features = features
        assert self.features in ('button', 'chamber', 'all'), "features argument must be 'button', 'chamber', or'all'."

        self.reference_images = {dname: None for dname in self.experiment.devices}
        
        # print('Loaded the following images:')
        # for image in image_data['image_path']:
        #     print(image)

    def _update_reference_image(self, dname: str, chip_image: ChipImage):
        self.reference_images[dname] = chip_image

    def _process_features(self, chip_image: ChipImage, coerce_chamber_center: bool = False):
        """Extract feature processing logic into a reusable method."""
        if self.features == 'chamber':
            chip_image.findChambers(coerce_center=coerce_chamber_center)
        elif self.features == 'button':
            chip_image.findButtons()
        elif self.features == 'all':
            chip_image.findChambers(coerce_center=coerce_chamber_center)
            chip_image.findButtons()

    def _save_summary_image_if_needed(self, chip_image: ChipImage, summary_image_path: Union[str, Path]):
        """Save summary image if path is provided."""
        if summary_image_path:
            skimage.io.imsave(summary_image_path, chip_image.summary_image(stamptype=self.features))

    def set_reference(
        self, 
        image: Union[Path, str], 
        corners: Tuple[tuple], 
        summary_image_path: Union[str, Path] = None, 
        coerce_chamber_center: bool = False
    ):
        """Set reference image for a device."""
        image = Path(image) if not isinstance(image, Path) else image

        # Get device number via metadata lookup 
        data = self.image_data[self.image_data['image_path'] == image]
        assert len(data) == 1, 'Duplicate images found!'
        dname = data.iloc[0]['dname']

        # Create and process chip image
        chip_image = ChipImage(self.experiment.devices[dname], image, corners)
        chip_image.stamp()
        self._process_features(chip_image, coerce_chamber_center)
        
        self._update_reference_image(dname, chip_image)
        self._save_summary_image_if_needed(chip_image, summary_image_path)

    def process(
        self, 
        *, 
        corners: Union[List[Tuple[tuple]], Tuple[tuple]] = None, 
        use_reference: bool = False, 
        summary_image_dir: Union[str, Path] = None, 
        coerce_chamber_center: bool = False, 
        **kwargs
    ):
        """High-level dispatcher that handles the mutual exclusivity logic."""
        if use_reference:
            if corners:
                raise ValueError("Cannot provide manual inputs when use_reference=True.")
            return self._process_from_reference(summary_image_dir=summary_image_dir, **kwargs)
        
        if corners:

            if not isinstance(corners, list):
                corners = [corners] * len(self.image_data)
            assert len(corners) == len(self.image_data), "Number of corner objects must equal number of images."

            return self._process_manually(corners, coerce_chamber_center=coerce_chamber_center, summary_image_dir=summary_image_dir, **kwargs)
        
        raise ValueError("Must provide either manual inputs (corners/features) or use_reference=True.")

    def _process_manually(
        self, 
        corners: Tuple[tuple], 
        summary_image_dir: Union[str, Path] = None, 
        coerce_chamber_center: bool = False
    ):
        """Process images with manually provided corners."""
        data = []
        for i, c in tqdm(enumerate(corners), desc='Processing images'):

            dname, image = self.image_data[['dname', 'image_path']].iloc[i]
            chip_image = ChipImage(self.experiment.devices[dname], image, c)
            chip_image.stamp()
            self._process_features(chip_image, coerce_chamber_center)

            processed_data = chip_image.summarize()
            metadata = pd.DataFrame([self.image_data.iloc[i]] * len(processed_data))
            merged = pd.concat([metadata, processed_data], axis=1)
            data.append(merged)
            
            if summary_image_dir:
                self._save_summary_image(summary_image_dir, image, chip_image)
        
        return pd.concat(data, ignore_index=False)

    def _process_from_reference(
        self, 
        summary_image_dir: Union[str, Path] = None
    ):
        """Process images by mapping from reference images."""

        data = []
        for i in tqdm(range(len(self.image_data)), desc='Processing images'):

            dname, image = self.image_data[['dname', 'image_path']].iloc[i]

            reference = self.reference_images.get(dname)
            if reference is None:
                raise ValueError(f"No reference image set for device {dname}. Call set_reference() first.")
            
            chip_image = ChipImage(self.experiment.devices[dname], image, reference.corners)
            chip_image.stamp()
            reference.mapto(chip_image, features=self.features)
            
            processed_data = chip_image.summarize().reset_index(drop=True)
            metadata = pd.DataFrame([self.image_data.iloc[i]] * len(processed_data)).reset_index(drop=True)
            # print(set(list(processed_data.columns)).intersection(set(list(metadata.columns))))
            merged = pd.concat([metadata, processed_data], axis=1)
            data.append(merged)
            
            if summary_image_dir:
                self._save_summary_image(summary_image_dir, image, chip_image, self.features)

        return pd.concat(data, ignore_index=False)

    def _save_summary_image(
        self, 
        summary_image_dir: Union[Path, str], 
        image_path: Union[Path, str], 
        chip_image: ChipImage, 
        features: str
    ):
        """Save summary images for debugging/review."""
        summary_image_dir = Path(summary_image_dir) if not isinstance(summary_image_dir, Path) else summary_image_dir
        image_path = Path(image_path) if not isinstance(image_path, Path) else image_path
        summary_image_dir.mkdir(parents=True, exist_ok=True)  # Fixed: removed redundant argument

        base_name = image_path.with_suffix('').name
        
        if features in ('button', 'all'):
            summary_image = chip_image.summary_image(stamptype='button')
            path = summary_image_dir / f'{base_name}_ButtonSummaryImage.tif'
            skimage.io.imsave(path, summary_image)

        if features in ('chamber', 'all'):
            summary_image = chip_image.summary_image(stamptype='chamber')
            path = summary_image_dir / f'{base_name}_ChamberSummaryImage.tif'
            skimage.io.imsave(path, summary_image)

General processing logic:
1. Grab images and associated metadata using `DataHandler` class methods.
2. Initialize a `Processor` object with the images / metadata from above
3. Process the data
    - Optionally, one can set and process a reference image. Then, features can be mapped to all other images in the series

## Button Quant Processing

In [ ]:
button_quant_image = data_handler.get_images(identifier='2026-02-10_19-20-21_d1_aura_cyan_2_5_sensitivity_2x2_1_expression_1hr')
bq_processor = Processor(experiment, image_data=button_quant_image)

Loaded the following images:
/Volumes/JSZ2/20260210/bgsub_images/2026-02-10_19-20-21_d1_aura_cyan_2_5_sensitivity_2x2_1_expression_1hr.tif


In [59]:
corners =((382, 589)), ((6541, 429)), ((527, 6906)), ((6812, 6663))
bq_processor.set_reference(
    image='/Volumes/JSZ2/20260210/bgsub_images/2026-02-10_19-20-21_d1_aura_cyan_2_5_sensitivity_2x2_1_expression_1hr.tif',
    corners=corners,
    features='button',
    summary_image_path='/Users/jonathanzhang/Desktop/test.tif'
)

1


/var/folders/kr/sy19m8_s16x4t46fc7wswfq80000gr/T/ipykernel_21521/1178046101.py:133: DeprecationWarning: `np.object` is a deprecated alias for the builtin `object`. To silence this warning, use `object` by itself. Doing this will not modify any behavior and is safe. 
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  a = np.empty((xdim, ydim), dtype=np.object)
Finding Buttons:   0%|          | 0/1792 [00:00<?, ?it/s]/var/folders/kr/sy19m8_s16x4t46fc7wswfq80000gr/T/ipykernel_21521/1178046101.py:539: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  mask = mask.astype(np.bool)
Finding Buttons: 100%|███

## Standard Curve Processing

In [166]:
warnings.filterwarnings("ignore", category=DeprecationWarning)

standard_images = data_handler.get_images(identifier='2026-03-05_12-26-07_standard_curve_NADH')

standard_processor = Processor(experiment, image_data=standard_images, features='chamber')
corners = ((458, 438)), ((6645, 437)), ((460, 6714)), ((6652, 6718))

# set references (for both devices)
standard_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_12-26-07_standard_curve_NADH/2026-03-05_13-11-23_d1_aura_UV340_5_1000_dynamic_range_2x2_4_standard_curve_NADH_7.tif', 
    corners=corners, 
    summary_image_path=root / 'standard_curve_chamber_summary_d1.tif'
    )
standard_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_12-26-07_standard_curve_NADH/2026-03-05_13-12-31_d2_aura_UV340_5_1000_dynamic_range_2x2_4_standard_curve_NADH_7.tif', 
    corners=corners, 
    summary_image_path=root / 'standard_curve_chamber_summary_d2.tif'
    )

In [167]:
standard_df = standard_processor.process(use_reference=True, summary_image_dir=root / 'standard_curve_chamber_summaries')

Processing images: 100%|██████████| 32/32 [01:36<00:00,  3.03s/it]


In [168]:
# add your desired output path here
standard_df.to_csv(root / 'standard_data.csv.bz2', compression='bz2')

## Kinetics Processing

In [170]:
warnings.filterwarnings("ignore", category=DeprecationWarning)

kinetics_images = data_handler.get_images(identifier='2026-03-05_13-17-36_kinetic_series_cycling')

kinetics_processor = Processor(experiment, image_data=kinetics_images, features='chamber')
corners = ((457, 437)), ((6644, 434)), ((464, 6716)), ((6649, 6714))

# set references (for both devices)
kinetics_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_13-17-36_kinetic_series_cycling/2026-03-05_13-17-36_timecourse_0/2026-03-05_13-17-49_d1_aura_UV340_5_1000_dynamic_range_2x2_4_kinetic_series_cycling_timecourse_0_0.tif', 
    corners=corners, 
    summary_image_path=root / 'kinetics_chamber_summary_d1.tif'
    )
kinetics_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_13-17-36_kinetic_series_cycling/2026-03-05_13-17-36_timecourse_0/2026-03-05_13-18-57_d2_aura_UV340_5_1000_dynamic_range_2x2_4_kinetic_series_cycling_timecourse_0_0.tif', 
    corners=corners, 
    summary_image_path=root / 'kinetics_chamber_summary_d2.tif'
    )

In [171]:
kinetics_df = kinetics_processor.process(use_reference=True, summary_image_dir=root / 'kinetics_chamber_summaries')

Processing images: 100%|██████████| 24/24 [01:10<00:00,  2.93s/it]


In [172]:
# add your desired output path here
kinetics_df.to_csv(root / 'kinetics_data.csv.bz2', compression='bz2')